# Setup and Imports

In [2]:
from typing import Annotated
from pydantic import BaseModel,Field
from dotenv import load_dotenv
from libs_4.messages import UserMessage, SystemMessage
from libs_4 import tool
import os
from libs_4.llm import LLM
from libs_4 import (
    StrOutputParser,
    JsonOutputParser,
    PydanticOutputParser,
    ToolOutputParser
)

In [3]:
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
print("API Key loaded:", api_key)
base_url = os.getenv("OPENAI_BASE_URL")
print("Base URL loaded:", base_url)

API Key loaded: voc-10734273616886548009516a1e9a3e1ef195.07556657
Base URL loaded: https://openai.vocareum.com/v1


In [7]:
chat_model = LLM(api_key=api_key, base_url=base_url)

## Basic String Output
Before diving into more complex output formats, let's understand how to work with simple string outputs from our Language Model.
This demonstrates the most basic form of parsing LLM responses.

In [8]:
messages = [
    SystemMessage(content="Extract the event information."),
    UserMessage(content="Alice and Bob are going to a science fair on Friday.")
]

2. **Basic String Output Parsing**
## #Start with a simple example to understand how to parse string outputs from the LLM. This step demonstrates the most basic form of output handling.

In [9]:
ai_message = chat_model.invoke(messages)
print("AI Message Content:\n", ai_message.content)
parser = StrOutputParser()
parsed_output = parser.parse(ai_message)
print("Parsed Output:\n", parsed_output)

AI Message Content:
 Event: Science Fair  
Participants: Alice and Bob  
Day: Friday
Parsed Output:
 Event: Science Fair  
Participants: Alice and Bob  
Day: Friday


3. **Working with Tools for Structured Outputs**
### Next, utilize tools to enforce a specific output format. This approach simplifies programmatic processing of the LLM's responses.

In [10]:
@tool
def calendar_event(name:str, date:str, participants:list[str]):
    """Identify name of the event, date when it will happen and all the participants"""
    return {
        "name": name,
        "date": date,
        "participants": participants
    }
chat_model_with_tools = LLM(api_key=api_key, base_url=base_url, tools=[calendar_event])
ai_message = chat_model_with_tools.invoke(messages)
parser = ToolOutputParser()
parsed_output = parser.parse(ai_message)[0]["args"]
print("Parsed Output:\n", parsed_output)

Parsed Output:
 {'name': 'Science Fair', 'date': 'Friday', 'participants': ['Alice', 'Bob']}


4. **Using Pydantic Models for Validation**
### Implement Pydantic models to validate and structure the outputs from the LLM. This step ensures type safety and data integrity.

In [11]:
class CalendarEvent(BaseModel):
    name: str = Annotated[str,Field(description="Name/Title of the event. Defaults to ''", default=None)]
    date: str = Annotated[str, Field(description="Date of the event. Defaults to ''", default=None)]
    participants: Annotated[list[str], Field(description="Who will participate. Defaults to ''", default=None)]
ai_message  = chat_model.invoke(input=messages, response_format=CalendarEvent)
parser = JsonOutputParser()
parsed_output = parser.parse(ai_message)
print("Parsed Output:\n", parsed_output)

C:\shiva\work\data_science\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value typing.Annotated[str, FieldInfo(annotation=NoneType, required=False, default=None, description="Name/Title of the event. Defaults to ''")] is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
C:\shiva\work\data_science\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value typing.Annotated[str, FieldInfo(annotation=NoneType, required=False, default=None, description="Date of the event. Defaults to ''")] is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


Parsed Output:
 {'name': 'Science Fair', 'date': 'Friday', 'participants': ['Alice', 'Bob']}


In [12]:
parser = PydanticOutputParser(model_class=CalendarEvent)
event: CalendarEvent = parser.parse(ai_message)
print("Parsed Event:\n", event)

Parsed Event:
 name='Science Fair' date='Friday' participants=['Alice', 'Bob']


In [13]:
participants = event.participants
print(participants)

['Alice', 'Bob']
